# GE338 Lab 3: Land Use / Land Cover Classification  
### Machine Learning with Sentinel-2 Imagery (Ayutthaya, Thailand)

In [2]:
!pip install -q earthengine-api geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.3 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import json

geojson_path = '/content/drive/MyDrive/ayutthaya_training_polygons_drive.geojson'

with open(geojson_path, 'r') as f:
    geojson_obj = json.load(f)

print('Loaded GeoJSON')
print('Number of features:', len(geojson_obj['features']))

Loaded GeoJSON
Number of features: 67


In [5]:
import ee
import geemap

# Authenticate and initialize
try:
    ee.Initialize(project='ee-natthanichap')
except:
    ee.Authenticate()
    ee.Initialize(project='ee-natthanichap')

In [6]:
training_polygons = ee.FeatureCollection(geojson_obj)

print('Training polygons loaded')
print('Number of features:', training_polygons.size().getInfo())

Training polygons loaded
Number of features: 67


In [7]:
roi = training_polygons.geometry().bounds().buffer(1000)

print('ROI ready')

ROI ready


In [8]:
def mask_s2_clouds(image):
    scl = image.select('SCL')
    mask = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )
    return image.updateMask(mask)

In [9]:
s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2025-11-01', '2026-02-28')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds)
    .median()
    .clip(roi)
)

print('Sentinel-2 composite ready')

Sentinel-2 composite ready


In [10]:
ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi = s2.normalizedDifference(['B3', 'B8']).rename('NDWI')
ndbi = s2.normalizedDifference(['B11', 'B8']).rename('NDBI')
ndmi = s2.normalizedDifference(['B8', 'B11']).rename('NDMI')

image = s2.addBands([ndvi, ndwi, ndbi, ndmi])

bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI', 'NDBI', 'NDMI']

print('Bands ready:', bands)

Bands ready: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI', 'NDBI', 'NDMI']


In [11]:
samples = image.select(bands).sampleRegions(
    collection=training_polygons,
    properties=['class'],
    scale=10,
    geometries=True
)

print('Number of samples:', samples.size().getInfo())

Number of samples: 7846


In [12]:
samples = samples.randomColumn(columnName='random', seed=42)

train = samples.filter(ee.Filter.lt('random', 0.8))
test = samples.filter(ee.Filter.gte('random', 0.8))

print('Train size:', train.size().getInfo())
print('Test size:', test.size().getInfo())

Train size: 6257
Test size: 1589


In [13]:
rf = ee.Classifier.smileRandomForest(numberOfTrees=100).train(
    features=train,
    classProperty='class',
    inputProperties=bands
)

print('Random Forest trained')

rf_test = test.classify(rf)
cm = rf_test.errorMatrix('class', 'classification')

print('Overall Accuracy:', cm.accuracy().getInfo())
print('Kappa:', cm.kappa().getInfo())
print('Confusion Matrix:')
print(cm.getInfo())
print('Producers Accuracy:')
print(cm.producersAccuracy().getInfo())
print('Users Accuracy:')
print(cm.consumersAccuracy().getInfo())



Random Forest trained
Overall Accuracy: 0.9955947136563876
Kappa: 0.9924618756463621
Confusion Matrix:
[[771, 0, 0, 0, 0], [1, 100, 1, 0, 1], [0, 1, 600, 0, 3], [0, 0, 0, 25, 0], [0, 0, 9, 3, 36]]
Producers Accuracy:
[[1], [0.9724770642201835], [0.9951298701298701], [0.96875], [1]]
Users Accuracy:
[[1, 0.9906542056074766, 0.993517017828201, 1, 0.9166666666666666]]


In [14]:
classified = image.select(bands).classify(rf)

print('Classification image ready')

Classification image ready


In [15]:
importance = ee.Dictionary(rf.explain().get('importance')).getInfo()

print('Feature Importance:')
print(importance)

Feature Importance:
{'B11': 141.32896623375285, 'B12': 157.64958555896115, 'B2': 133.80663229273057, 'B3': 128.991857523752, 'B4': 147.24540058654262, 'B8': 130.09407423024405, 'NDBI': 96.10613387426548, 'NDMI': 103.58327916104085, 'NDVI': 150.35050829502842, 'NDWI': 139.89393597553854}


In [16]:
Map = geemap.Map(center=[14.35, 100.60], zoom=11)

vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
class_vis = {
    'min': 0,
    'max': 4,
    'palette': ['0000FF', '00A600', 'FFD700', 'B04CC2', 'A87000']
}

Map.addLayer(s2, vis_rgb, 'Sentinel-2 RGB')
Map.addLayer(classified, class_vis, 'RF Classification')
Map.addLayer(training_polygons, {}, 'Training Polygons')

Map

Map(center=[14.35, 100.6], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…

In [17]:
svm = ee.Classifier.libsvm().train(
    features=train,
    classProperty='class',
    inputProperties=bands
)

svm_test = test.classify(svm)
svm_cm = svm_test.errorMatrix('class', 'classification')

print('SVM Overall Accuracy:', svm_cm.accuracy().getInfo())
print('SVM Kappa:', svm_cm.kappa().getInfo())
print('SVM Confusion Matrix:')
print(svm_cm.getInfo())

SVM Overall Accuracy: 0.6859660163624921
SVM Kappa: 0.5362887473361388
SVM Confusion Matrix:
[[769, 1, 1, 0, 0], [0, 101, 1, 0, 1], [0, 2, 602, 0, 0], [0, 0, 0, 25, 0], [0, 3, 41, 4, 0]]


In [20]:
bands_simple = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

rf_simple = ee.Classifier.smileRandomForest(numberOfTrees=100).train(
    features=train,
    classProperty='class',
    inputProperties=bands_simple
)

rf_simple_test = test.classify(rf_simple)
cm_simple = rf_simple_test.errorMatrix('class', 'classification')

print('RF Bands Only Accuracy:', cm_simple.accuracy().getInfo())
print('RF Bands Only Kappa:', cm_simple.kappa().getInfo())

RF Bands Only Accuracy: 0.9962240402769037
RF Bands Only Kappa: 0.9782930894769769


In [21]:
rf_full = ee.Classifier.smileRandomForest(numberOfTrees=100).train(
    features=train,
    classProperty='class',
    inputProperties=bands
)

rf_full_test = test.classify(rf_full)
cm_full = rf_full_test.errorMatrix('class', 'classification')

print('RF Bands + Indices Accuracy:', cm_full.accuracy().getInfo())
print('RF Bands + Indices Kappa:', cm_full.kappa().getInfo())

RF Bands + Indices Accuracy: 0.9955947136563876
RF Bands + Indices Kappa: 0.9924618756463621


In [27]:
rf_multi_prob = ee.Classifier.smileRandomForest(numberOfTrees=100) \
    .setOutputMode('MULTIPROBABILITY') \
    .train(
        features=train,
        classProperty='class',
        inputProperties=bands
    )

multi_prob = image.select(bands).classify(rf_multi_prob)

max_prob = multi_prob.arrayReduce(ee.Reducer.max(), [0]) \
    .arrayGet([0]) \
    .rename('max_prob')

uncertainty = ee.Image.constant(1).subtract(max_prob).rename('uncertainty')

Map2 = geemap.Map(center=[14.35, 100.60], zoom=11)
Map2.addLayer(max_prob, {'min': 0, 'max': 1}, 'Max Probability')
Map2.addLayer(uncertainty, {'min': 0, 'max': 1}, 'Uncertainty')
Map2

Map(center=[14.35, 100.6], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…